# ClaimLens — Error analysis

The evaluation report gives you the *score*. This notebook tells you **why** the score is what it is:
which fields fail, what kind of mistake the model makes, which document conditions cause it, and whether errors
fall unevenly across (synthetic) name groups.

**How to use:** set `RUN_DIR` below to any run folder (e.g. the baseline you ran on Kaggle), then *Run All*.
Runs on a laptop CPU.

> If `RUN_DIR` points at a **simulated-predictor** run, every pattern you see was planted on purpose (see
> `claimlens/predictors.py`). That is useful for checking the notebook works — but only findings from a real model
> run belong in your write-up.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

RUN_DIR = ROOT / "runs" / "sim_test"   # <-- change me

config = json.loads((RUN_DIR / "config.json").read_text())
DATA_DIR = Path(config["data_dir"])
if not DATA_DIR.is_absolute():
    DATA_DIR = ROOT / DATA_DIR
SPLIT_DIR = DATA_DIR / config["split"]
df = pd.read_json(RUN_DIR / "scored_fields.jsonl", lines=True)
docs = pd.read_json(RUN_DIR / "results.jsonl", lines=True)
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
print(f"Run {config['run_id']}  |  model {config['model_id']}  |  {docs.shape[0]} docs, {df.shape[0]} field predictions")
if config["model_id"] == "simulated-predictor":
    print("NOTE: simulated predictor -- patterns below were planted, not discovered.")

In [ ]:
# Chart style: one hue for magnitude, quiet axes, values labelled at the bar end.
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#d9d8d4",
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK, "axes.grid": True,
                     "grid.color": "#ecebe8", "grid.linewidth": 0.8, "axes.axisbelow": True,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def barh(series, title, ax=None, pct=True, xmax=None):
    s = series.sort_values()
    ax = ax or plt.subplots(figsize=(7, 0.42 * len(s) + 1))[1]
    ax.barh(s.index.astype(str), s.values * (100 if pct else 1), color=BLUE, height=0.6)
    for i, v in enumerate(s.values):
        ax.text(v * (100 if pct else 1), i, f"  {v*100:.0f}%" if pct else f"  {v:,.0f}", va="center",
                color=INK, fontsize=9)
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_xlim(0, xmax or ((105 if pct else s.max() * 1.15)))
    ax.grid(axis="y", visible=False)
    if pct: ax.set_xlabel("%")
    return ax

## 1. Which fields fail?
Accuracy counts a correctly-returned `null` as correct. *Grounded* accuracy also requires the citation box to be in the right place.

In [ ]:
acc = df.groupby("field")["correct"].mean()
present = df[df.gold_present]
grounded = present.assign(g=present.correct & present.citation_ok).groupby("field")["g"].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=False)
barh(acc, "Value accuracy by field", axes[0])
barh(grounded, "Grounded accuracy by field (value + citation)", axes[1])
plt.tight_layout(); plt.show()

## 2. What kind of mistakes?
| Error type | Meaning |
|---|---|
| `near_miss` | A character or two off — an OCR-style misread |
| `distractor_confusion` | Read a *different* value that is also on the page (subtotal for total, report date for loss date) |
| `field_swap` | Put another field's value here (policy number as claim number) |
| `diacritics` | Dropped accents (José → Jose) |
| `missed` | Returned null although the field is on the page |
| `hallucinated` | Invented a value for a field that is not on the page |
| `unparseable` | Output was not valid JSON, so every field is lost |
| `wrong_value` | Anything else |

In [ ]:
errs = df[~df.correct]
counts = errs.error_type.value_counts()
barh(counts / counts.sum(), f"Share of all {len(errs)} errors by type"); plt.show()
pd.crosstab(errs.field, errs.error_type, margins=True)

## 3. Which document conditions drive errors?
The **lift** column is the key: a slice holding 20% of fields but 45% of errors has lift 2.25 — that is where to focus data collection or training.

In [ ]:
def concentration(col):
    t = df.groupby(col).agg(fields=("correct", "size"), errors=("correct", lambda s: (~s).sum()))
    t["share_of_fields"] = t.fields / t.fields.sum()
    t["share_of_errors"] = t.errors / max(1, t.errors.sum())
    t["lift"] = (t.share_of_errors / t.share_of_fields).round(2)
    t["accuracy"] = 1 - t.errors / t.fields
    return t.sort_values("lift", ascending=False)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, col in zip(axes, ["quality", "handwritten", "doc_type", "line_of_business"]):
    barh(df.groupby(col)["correct"].mean(), f"Accuracy by {col}", ax)
plt.tight_layout(); plt.show()
for col in ["quality", "handwritten", "doc_type"]:
    display(concentration(col).style.format({"share_of_fields": "{:.0%}", "share_of_errors": "{:.0%}", "accuracy": "{:.1%}"}))

## 4. Grounding: right answer, wrong place?
A correct value with a wrong citation is still a problem in a regulated setting — an auditor cannot verify it.

In [ ]:
right = df[df.correct & df.gold_present & df.pred_present]
print(f"Correct values whose citation is on target (IoU >= 0.5): {right.citation_ok.mean():.1%}")
print(f"Correct values whose cited box really contains the value: {right.evidence_supported.mean():.1%}")
ax = plt.subplots(figsize=(7, 2.8))[1]
ax.hist(right.iou, bins=20, range=(0, 1), color=BLUE, edgecolor="#fcfcfb", linewidth=2)
ax.axvline(0.5, color=MUTED, ls="--", lw=1); ax.text(0.51, ax.get_ylim()[1] * 0.9, "threshold", color=MUTED)
ax.set_title("IoU between cited box and true box (correct values only)", loc="left", color=INK)
ax.set_xlabel("IoU"); plt.show()

## 5. Fairness
Groups are synthetic name-origin proxies (see `claimlens/names.py`). Two questions:
1. Is accuracy similar across groups — for the name itself and for every *other* field?
2. On the `fairness` split: when **only** the name changes, do the other fields stay the same?

In [ ]:
f = metrics["fairness"]
by_group = pd.DataFrame(f["by_group"]).T
display(by_group.style.format({"other_fields_accuracy": "{:.1%}", "name_accuracy": "{:.1%}"}))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
barh(by_group.name_accuracy.astype(float), "Claimant-name accuracy by group", axes[0])
barh(by_group.other_fields_accuracy.astype(float), "Other-field accuracy by group", axes[1])
plt.tight_layout(); plt.show()
print(f"Group gap (name): {f['group_gap_name']}, (other fields): {f['group_gap_other_fields']}")
if f["counterfactual_sets"]:
    print(f"Counterfactual consistency: {f['counterfactual_consistency']:.1%} of {f['counterfactual_sets']} sets")
    print("Flip rate by field:", {k: v for k, v in f["counterfactual_flip_rate_by_field"].items() if v})
else:
    print("Run this notebook on a fairness-split run to see counterfactual consistency.")

## 6. Look at real failures
Numbers tell you *where*; pictures tell you *why*. Green = true location, orange = where the model pointed.

In [ ]:
_v = lambda x: x if isinstance(x, str) else None

def show_failures(n=6, error_type=None, field=None, seed=0):
    pool = errs.copy()
    if error_type: pool = pool[pool.error_type == error_type]
    if field: pool = pool[pool.field == field]
    pool = pool[pool.error_type != "unparseable"]
    if pool.empty:
        print("No matching failures."); return
    sample = pool.sample(min(n, len(pool)), random_state=seed)
    fig, axes = plt.subplots(len(sample), 1, figsize=(10, 2.3 * len(sample)))
    axes = [axes] if len(sample) == 1 else axes
    for ax, (_, r) in zip(axes, sample.iterrows()):
        img = Image.open(SPLIT_DIR / "images" / f"{r.doc_id}.jpg").convert("RGB")
        d = ImageDraw.Draw(img)
        boxes = [b for b in [r.gold_bbox, r.pred_bbox] if isinstance(b, list)]
        if r.gold_bbox is not None and isinstance(r.gold_bbox, list): d.rectangle(r.gold_bbox, outline=AQUA, width=6)
        if r.pred_bbox is not None and isinstance(r.pred_bbox, list): d.rectangle(r.pred_bbox, outline=ORANGE, width=2)
        if boxes:
            y0 = max(0, min(b[1] for b in boxes) - 60); y1 = min(img.height, max(b[3] for b in boxes) + 60)
            img = img.crop((0, y0, img.width, y1))
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"{r.doc_id} · {r.field} · {r.error_type}   gold={_v(r.gold_value)!r}   pred={_v(r.pred_value)!r}",
                     loc="left", fontsize=9, color=INK)
    plt.tight_layout(); plt.show()

show_failures(n=6)

In [ ]:
# Drill into one failure mode, e.g.:
show_failures(n=4, error_type="distractor_confusion")

## 7. Findings (data-generated starting point)
These sentences are computed from the run — edit them into your own words, and only keep what holds up when you look at the examples.

In [ ]:
lines = []
top = counts.index[0] if len(counts) else None
if top:
    lines.append(f"- The most common error type is `{top}` ({counts.iloc[0] / counts.sum():.0%} of {counts.sum()} errors).")
worst = acc.idxmin()
lines.append(f"- The weakest field is `{worst}` ({acc.min():.1%} accuracy vs {acc.mean():.1%} average).")
for col in ["quality", "handwritten", "doc_type"]:
    c = concentration(col)
    top_slice = c.index[0]
    if c.lift.iloc[0] >= 1.3:
        lines.append(f"- `{col}={top_slice}` holds {c.share_of_fields.iloc[0]:.0%} of fields but "
                     f"{c.share_of_errors.iloc[0]:.0%} of errors (lift {c.lift.iloc[0]}).")
if len(right):
    lines.append(f"- {1 - right.citation_ok.mean():.0%} of correct values cite the wrong location.")
if f["group_gap_name"] is not None:
    lines.append(f"- Claimant-name accuracy varies by {f['group_gap_name']:.0%} across name groups.")
if f["counterfactual_sets"]:
    lines.append(f"- Changing only the name changed another field in "
                 f"{1 - f['counterfactual_consistency']:.0%} of counterfactual sets.")
print("\n".join(lines))